In [2]:
# Cell 1: Imports and connection setup
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Make sure Python can find src/ from the notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent))

# src/ imports
from src.db import get_connection
from src.config import DATA_RAW, DATA_GENERATED
from src.generate_claims import build_providers, build_claim_header, build_claim_line
from src.generate_context import build_encounter_context
from src.generate_remittance import generate_remittance
from src.build_forensics import build_denial_forensics
from src.appeal_outcomes import simulate_appeal_outcomes

# Open a write connection to DuckDB
con = get_connection(read_only=False)

print("Connection established.")
print(f"Database path: {con}")
print(f"Raw data path: {DATA_RAW}")

Connection established.
Database path: <_duckdb.DuckDBPyConnection object at 0x10d3e70b0>
Raw data path: /Users/modelnic/coding/claims-leakage-audit/data/raw


In [3]:
# Cell 2: Execute schema — creates all empty tables in DuckDB
schema_path = Path.cwd().parent / "src" / "schema.sql"

with open(schema_path, "r") as f:
    schema_sql = f.read()

# Split on semicolons and execute each statement individually
statements = [s.strip() for s in schema_sql.split(";") if s.strip()]
for statement in statements:
    con.execute(statement)

# Verify tables were created
tables = con.execute("SHOW TABLES").fetchall()
print("Tables created:")
for t in tables:
    print(f"  {t[0]}")

Tables created:
  appeal_outcomes
  claim_header
  claim_line
  denial_forensics
  encounter_context
  member_eligibility
  provider
  remittance_835


In [4]:
# Cell 3: Load Synthea raw data
encounters_df  = pd.read_csv(DATA_RAW / "encounters.csv",  low_memory=False).sample(n=25000, random_state=42)
procedures_df  = pd.read_csv(DATA_RAW / "procedures.csv",  low_memory=False)
patients_df    = pd.read_csv(DATA_RAW / "patients.csv",    low_memory=False)
conditions_df  = pd.read_csv(DATA_RAW / "conditions.csv",  low_memory=False)

print(f"Encounters:  {len(encounters_df):,} rows")
print(f"Procedures:  {len(procedures_df):,} rows")
print(f"Patients:    {len(patients_df):,} rows")
print(f"Conditions:  {len(conditions_df):,} rows")

print("Encounters columns:")
print(list(encounters_df.columns))

# Quick sanity check — confirm expected columns exist
assert "PATIENT" in encounters_df.columns, "Missing PATIENT column in encounters"
assert "DATE" in encounters_df.columns, "Missing DATE column in encounters"
assert "CODE"    in procedures_df.columns, "Missing CODE column in procedures"
print("\nColumn checks passed.")

Encounters:  25,000 rows
Procedures:  628,785 rows
Patients:    132,896 rows
Conditions:  483,462 rows
Encounters columns:
['ID', 'DATE', 'PATIENT', 'CODE', 'DESCRIPTION', 'REASONCODE', 'REASONDESCRIPTION']

Column checks passed.


In [5]:
# Cell 4: Build provider table from personas
providers_df = build_providers()

print(f"Providers built: {len(providers_df)} rows")
print(providers_df[["provider_id", "specialty", "persona_type", "network_status"]])

# Load into DuckDB
con.execute("DELETE FROM provider")  # clear if re-running
con.execute("INSERT INTO provider SELECT * FROM providers_df")

count = con.execute("SELECT COUNT(*) FROM provider").fetchone()[0]
print(f"\nProvider table: {count} rows in DuckDB")

Providers built: 5 rows
  provider_id           specialty               persona_type network_status
0    PROV_001   Internal Medicine        slow_charge_capture     in-network
1    PROV_002           Radiology            prior_auth_miss     in-network
2    PROV_003  Emergency Medicine              coding_errors     in-network
3    PROV_004   Behavioral Health       credentialing_errors     in-network
4    PROV_005            Oncology  medical_necessity_denials     in-network

Provider table: 5 rows in DuckDB


In [6]:
# Cell 5: Build claim_header from Synthea encounters
claim_header_df = build_claim_header(encounters_df, providers_df)

print(f"Claim headers built: {len(claim_header_df):,} rows")
print("\nSample:")
print(claim_header_df.head(3).to_string())

# Distribution check — payer weights should reflect 45/30/20/5 split
print("\nPayer distribution:")
print(claim_header_df["payer_id"].value_counts(normalize=True).round(3))

# Load into DuckDB
con.execute("DELETE FROM claim_header")
con.execute("INSERT INTO claim_header SELECT * FROM claim_header_df")

count = con.execute("SELECT COUNT(*) FROM claim_header").fetchone()[0]
print(f"\nclaim_header table: {count:,} rows in DuckDB")

Claim headers built: 25,000 rows

Sample:
                               claim_id                            patient_id provider_id            payer_id date_of_service submission_date claim_type place_of_service  billed_amount claim_status
0  59c6c2f7-d1be-47a4-b9a1-5658e96295eb  2ba9c8f1-1a3e-48ef-9664-f88b0a41a31d    PROV_002      commercial_ppo      2010-07-13      2010-07-16       837P               11        1701.21    submitted
1  3fa78ee9-7d63-4be9-99e9-1390353183c0  27721d2b-c464-4bd5-9fa2-aea6d615e742    PROV_001      commercial_ppo      2012-07-07      2012-07-19       837P               11         248.69    submitted
2  0f7fa2fc-a01b-49d9-b1a3-788060b2b97e  b18a08de-05a8-42a6-9696-ffd3421acf1c    PROV_005  medicare_advantage      2012-06-04      2012-06-05       837P               11        7647.95    submitted

Payer distribution:
payer_id
medi_cal_managed_care     0.453
commercial_ppo            0.297
medicare_advantage        0.200
dual_eligible_cobpayer    0.049
Name: pr

In [7]:
# Cell 6: Build claim_line from Synthea procedures
claim_line_df = build_claim_line(claim_header_df, procedures_df)

print(f"Claim lines built: {len(claim_line_df):,} rows")
print("\nSample:")
print(claim_line_df.head(5).to_string())

# CPT distribution — should show variety, not all fallback 99213
print("\nTop 10 CPT codes:")
print(claim_line_df["cpt_code"].value_counts().head(10))

# Load into DuckDB
con.execute("DELETE FROM claim_line")
con.execute("INSERT INTO claim_line SELECT * FROM claim_line_df")

count = con.execute("SELECT COUNT(*) FROM claim_line").fetchone()[0]
print(f"\nclaim_line table: {count:,} rows in DuckDB")

Claim lines built: 69,574 rows

Sample:
                          claim_line_id                              claim_id  line_number cpt_code icd10_primary  units  billed_amount line_status
0  864d8eb8-eaea-43e4-bdf6-334f188d4068  59c6c2f7-d1be-47a4-b9a1-5658e96295eb            1    99213        Z00.00      1        1701.21   submitted
1  c9220256-869b-4d11-a420-478b05cbb0cb  59c6c2f7-d1be-47a4-b9a1-5658e96295eb            2    99213        Z00.00      1         850.61   submitted
2  d5243bab-77f3-4419-90fa-d994ae036e27  3fa78ee9-7d63-4be9-99e9-1390353183c0            1    99213        Z00.00      1         248.69   submitted
3  ab2b4760-df94-4c33-afdc-8e1aa43f2c2f  3fa78ee9-7d63-4be9-99e9-1390353183c0            2    99213        Z00.00      1         124.34   submitted
4  363df0cc-c4a9-4f64-ac48-7d2990fda86d  3fa78ee9-7d63-4be9-99e9-1390353183c0            3    12001        Z00.00      1          82.90   submitted

Top 10 CPT codes:
cpt_code
99213    37836
45378     4619
94010     3418

In [8]:
# Cell 7: Build encounter_context — the upstream decision state
encounter_context_df = build_encounter_context(claim_header_df, claim_line_df, providers_df)

print(f"Encounter contexts built: {len(encounter_context_df):,} rows")

# Sanity checks — rates should reflect persona parameters
print("\nEligibility verification rate:")
print(round(encounter_context_df["eligibility_verified_flag"].mean(), 3))

print("\nPrior auth required rate:")
print(round(encounter_context_df["prior_auth_required"].mean(), 3))

print("\nCoding accuracy rate:")
print(round(encounter_context_df["coding_accuracy_flag"].mean(), 3))

print("\nCredentialing lapsed rate:")
print(round(encounter_context_df["credentialing_lapsed"].mean(), 3))

# Load into DuckDB
con.execute("DELETE FROM encounter_context")
con.execute("""
    INSERT INTO encounter_context
    SELECT 
        encounter_id,
        claim_id,
        patient_id,
        provider_id,
        date_of_service,
        eligibility_verified_flag,
        credentialing_lapsed,
        prior_auth_required,
        prior_auth_obtained,
        auth_request_date,
        charge_entry_lag_days,
        coding_accuracy_flag
    FROM encounter_context_df
""")

count = con.execute("SELECT COUNT(*) FROM encounter_context").fetchone()[0]
print(f"\nencounter_context table: {count:,} rows in DuckDB")

Encounter contexts built: 25,000 rows

Eligibility verification rate:
0.824

Prior auth required rate:
0.266

Coding accuracy rate:
0.881

Credentialing lapsed rate:
0.023

encounter_context table: 25,000 rows in DuckDB


In [9]:
# Cell 8: Generate remittance_835 — payer adjudication
remittance_df = generate_remittance(claim_line_df, claim_header_df, encounter_context_df)

print(f"Remittance rows built: {len(remittance_df):,} rows")

# Denial rate check — should be non-uniform, not 0% or 100%
denied = remittance_df[remittance_df["paid_amount"] == 0]
denial_rate = len(denied) / len(remittance_df)
print(f"\nOverall denial rate: {denial_rate:.1%}")

# CARC distribution — should show multiple codes
print("\nCARCs issued:")
print(remittance_df["carc_code"].value_counts())

# Financial constraint check — verify accounting identity holds
# For paid rows: allowed = paid + patient_responsibility
paid_rows = remittance_df[remittance_df["paid_amount"] > 0]
violations = paid_rows[
    abs(paid_rows["allowed_amount"] - (
        paid_rows["paid_amount"] +
        paid_rows["patient_responsibility"]
    )) >= 0.01
]
print(f"\nFinancial constraint violations: {len(violations)} (should be 0)")

# Load into DuckDB
con.execute("DELETE FROM remittance_835")
con.execute("INSERT INTO remittance_835 SELECT * FROM remittance_df")

count = con.execute("SELECT COUNT(*) FROM remittance_835").fetchone()[0]
print(f"\nremittance_835 table: {count:,} rows in DuckDB")

Remittance rows built: 69,574 rows

Overall denial rate: 15.3%

CARCs issued:
carc_code
CO-45    59485
CO-4      3276
CO-16     1591
CO-B7     1505
CO-57     1281
CO-50      759
CO-97      634
CO-22      595
CO-29      448
Name: count, dtype: int64

Financial constraint violations: 0 (should be 0)

remittance_835 table: 69,574 rows in DuckDB


In [10]:
# Cell 9: Build denial_forensics — the analytical layer
denial_forensics_df = build_denial_forensics(remittance_df, claim_header_df)

print(f"Denial forensics rows built: {len(denial_forensics_df):,} rows")

# Upstream failure node distribution — this should have shape
print("\nUpstream failure nodes:")
print(denial_forensics_df["upstream_failure_node"].value_counts())

# Preventability breakdown
print("\nPreventable vs non-preventable:")
print(denial_forensics_df["preventability_flag"].value_counts())

# Top priority denials by score
print("\nTop 5 priority denials:")
print(
    denial_forensics_df
    .sort_values("priority_score", ascending=False)
    .head(5)[["carc_code", "denial_category", "upstream_failure_node",
              "dollars_at_risk", "recovery_probability", "priority_score"]]
    .to_string()
)

# Load into DuckDB
con.execute("DELETE FROM denial_forensics")
con.execute("INSERT INTO denial_forensics SELECT * FROM denial_forensics_df")

count = con.execute("SELECT COUNT(*) FROM denial_forensics").fetchone()[0]
print(f"\ndenial_forensics table: {count:,} rows in DuckDB")

Denial forensics rows built: 10,089 rows

Upstream failure nodes:
upstream_failure_node
coding                 3910
prior_authorization    2040
billing                1591
credentialing          1505
remittance_posting      595
charge_capture          448
Name: count, dtype: int64

Preventable vs non-preventable:
preventability_flag
True     9330
False     759
Name: count, dtype: int64

Top 5 priority denials:
     carc_code      denial_category upstream_failure_node  dollars_at_risk  recovery_probability  priority_score
8058     CO-16  missing_information               billing         11924.76                  0.72      17171.6544
308      CO-16  missing_information               billing         11808.73                  0.72      17004.5712
6920     CO-16  missing_information               billing         11763.96                  0.72      16940.1024
2447      CO-4         coding_error                coding         11978.25                  0.70      16769.5500
4528     CO-16  missi

In [11]:
# Cell 10: Simulate appeal outcomes — recovery layer
appeal_outcomes_df = simulate_appeal_outcomes(denial_forensics_df, remittance_df)

print(f"Appeal outcomes simulated: {len(appeal_outcomes_df):,} rows")

# Recovery summary
total_at_risk   = appeal_outcomes_df["dollars_at_risk"].sum()
total_recovered = appeal_outcomes_df["dollars_recovered"].sum()
recovery_rate   = total_recovered / total_at_risk if total_at_risk > 0 else 0

print(f"\nTotal dollars at risk (appealed):  ${total_at_risk:,.2f}")
print(f"Total dollars recovered:            ${total_recovered:,.2f}")
print(f"Overall recovery rate:              {recovery_rate:.1%}")

# Load into DuckDB
con.execute("DELETE FROM appeal_outcomes")
con.execute("INSERT INTO appeal_outcomes SELECT * FROM appeal_outcomes_df")

count = con.execute("SELECT COUNT(*) FROM appeal_outcomes").fetchone()[0]
print(f"\nappeal_outcomes table: {count:,} rows in DuckDB")

Appeal outcomes simulated: 9,641 rows

Total dollars at risk (appealed):  $12,367,341.75
Total dollars recovered:            $6,557,399.28
Overall recovery rate:              53.0%

appeal_outcomes table: 9,641 rows in DuckDB


In [12]:
# Cell 11: Final summary — confirm all tables populated
print("=" * 50)
print("DATABASE BUILD COMPLETE")
print("=" * 50)

tables = [
    "provider", "member_eligibility", "claim_header",
    "claim_line", "remittance_835", "encounter_context",
    "denial_forensics", "appeal_outcomes"
]

for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<25} {count:>8,} rows")

# Close the write connection
con.close()
print("\nConnection closed. Database ready for analysis.")

DATABASE BUILD COMPLETE
  provider                         5 rows
  member_eligibility               0 rows
  claim_header                25,000 rows
  claim_line                  69,574 rows
  remittance_835              69,574 rows
  encounter_context           25,000 rows
  denial_forensics            10,089 rows
  appeal_outcomes              9,641 rows

Connection closed. Database ready for analysis.
